### Solving the Optimization Problem with Julia

Mixed-integer facility location model (p-median) for choosing **P = 5** ambulance standby sites.

$$
\min_{x,y}\ \frac{\sum_{i\in I}\sum_{j\in J} w_i\, t_{ji}\, x_{ji}}{\sum_{i\in I} w_i}
\quad \text{s.t.}\quad \sum_{j\in J} y_j = P,\qquad \sum_{j\in J} x_{ji} = 1\ \ (\forall i\in I),\qquad x_{ji}\le y_j\ \ (\forall i\in I,\ j\in J),\qquad x,y\in\{0,1\}
$$

- $I$: ZIP areas (demand points), $w_i$: call count in area $i$ during the planning period
- $J$: candidate standby sites, $t_{ji}$: estimated driving time (minutes) from site $j$ to area $i$'s representative point
- $y_j$: site $j$ is selected; $x_{ji}$: site $j$ serves area $i$

Data files:
- `data/demand.csv`: one row per Brooklyn ZIP area: `area_id, lat, lon, calls` (the area's representative point and its weekday-daytime call count, built in `Analysis.ipynb` / `Geography.ipynb`; ZIP 11385 already excluded). `data/demand_centroids.csv` has the same schema but uses polygon centroids instead.
- `data/candidate_sites.csv`: candidate standby sites $J$ (the 66 Brooklyn FDNY firehouses) with their `site_id`s. Built by `TravelTimes.ipynb`.
- `data/travel_times.csv`: $t_{ji}$ in long format `site_id, area_id, euclid_miles, network_miles, minutes` (shortest street-network route at a uniform speed). Built by `TravelTimes.ipynb`; run that notebook first.

In [1]:
using CSV, DataFrames, JuMP, Gurobi, Printf

#### 1. Configuration

In [3]:
DATA_DIR   = "data"
OUTPUT_DIR = "results"

DEMAND_FILE      = joinpath(DATA_DIR, "demand.csv")                          # area_id, lat, lon, calls
CANDIDATE_FILE   = joinpath(DATA_DIR, "candidate_sites.csv")                 # site_id, name, address, lat, lon
TRAVEL_TIME_FILE = joinpath(DATA_DIR, "travel_times_routed.csv")                   # site_id, area_id, ..., minutes

P          = 5               # number of standby sites to open
WEIGHT_COL = :calls          # w_i: call count per ZIP area
TIME_COL   = :minutes        # t_ji column in TRAVEL_TIME_FILE

TIME_LIMIT_SEC = 300

300

#### 2. Load demand areas ($I$, $w_i$)

In [4]:
demand = CSV.read(DEMAND_FILE, DataFrame)
demand.area_id = string.(demand.area_id)
dropmissing!(demand, [:lat, :lon, WEIGHT_COL])
filter!(row -> row[WEIGHT_COL] > 0, demand)

I_areas = demand.area_id
w = Float64.(demand[!, WEIGHT_COL])
println("Demand areas |I| = $(length(I_areas)),  total $(WEIGHT_COL) = $(sum(w))")
first(demand, 5)

Demand areas |I| = 37,  total calls = 202491.0


Row,area_id,lat,lon,calls
,String,Float64,Float64,Int64
1,11201,40.6946,-73.9903,11460
2,11203,40.6487,-73.9339,7205
3,11204,40.621,-73.9859,2877
4,11205,40.6965,-73.9664,3461
5,11206,40.7024,-73.9419,8664


#### 3. Load candidate standby sites ($J$)

In [5]:
candidates = CSV.read(CANDIDATE_FILE, DataFrame)
candidates.site_id = string.(candidates.site_id)

J_sites = candidates.site_id
println("Candidate sites |J| = $(length(J_sites))")
first(candidates, 5)

Candidate sites |J| = 66


Row,site_id,name,address,lat,lon,snap_miles
,String7,String,String31,Float64,Float64,Float64
1,FH_001,Brooklyn Borough Command/Division 11/Battalion 31/Engine 207/Ladder 110,172 Tillary Street,40.6955,-73.9828,0.0355387
2,FH_002,Engine 226,409 State Street,40.6871,-73.9828,0.0370268
3,FH_003,Engine 205/Ladder 118,74 Middagh Street,40.7,-73.9923,0.0269803
4,FH_004,Engine 224,274 Hicks Street,40.6931,-73.9971,0.031735
5,FH_005,Engine 279/Ladder 131,252 Lorraine St.,40.6721,-73.9997,0.0198208


#### 4. Travel times $t_{ji}$

`t[j, i]` = minutes from site `j` to area `i`, read from `TRAVEL_TIME_FILE` (built by `TravelTimes.ipynb`).

In [6]:
function load_travel_times(path, J_sites, I_areas; col=TIME_COL)
    tt = CSV.read(path, DataFrame)
    tt.site_id = string.(tt.site_id)
    tt.area_id = string.(tt.area_id)
    jidx = Dict(s => k for (k, s) in enumerate(J_sites))
    iidx = Dict(a => k for (k, a) in enumerate(I_areas))
    t = fill(NaN, length(J_sites), length(I_areas))
    for r in eachrow(tt)
        haskey(jidx, r.site_id) && haskey(iidx, r.area_id) || continue
        t[jidx[r.site_id], iidx[r.area_id]] = r[col]
    end
    n_missing = count(isnan, t)
    n_missing == 0 || error("$(n_missing) (site, area) pairs missing from $(path): rerun TravelTimes.ipynb")
    return t
end

t = load_travel_times(TRAVEL_TIME_FILE, J_sites, I_areas)

@printf("t_ji: %d × %d, range %.1f–%.1f min\n", size(t)..., minimum(t), maximum(t))

t_ji: 66 × 37, range 1.2–49.8 min


#### 5. Build and solve the MIP

In [7]:
function solve_standby_model(t, w, P; time_limit=TIME_LIMIT_SEC, silent=true)
    nJ, nI = size(t)
    model = Model(Gurobi.Optimizer)
    silent && set_silent(model)
    set_time_limit_sec(model, time_limit)

    @variable(model, y[1:nJ], Bin)          # site j selected
    @variable(model, x[1:nJ, 1:nI], Bin)    # site j serves area i

    @objective(model, Min, sum(w[i] * t[j, i] * x[j, i] for j in 1:nJ, i in 1:nI) / sum(w))

    @constraint(model, open_P, sum(y) == P)
    @constraint(model, assign[i in 1:nI], sum(x[j, i] for j in 1:nJ) == 1)
    @constraint(model, link[j in 1:nJ, i in 1:nI], x[j, i] <= y[j])

    optimize!(model)
    is_solved_and_feasible(model) || error("Solve failed: $(termination_status(model))")

    selected = findall(>(0.5), value.(y))
    assigned = [findfirst(>(0.5), value.(x[:, i])) for i in 1:nI]
    return (model=model, selected=selected, assigned=assigned, objective=objective_value(model))
end

sol = solve_standby_model(t, w, P)
println("Status: ", termination_status(sol.model), " | solve time: ",
        round(solve_time(sol.model), digits=2), " s")
@printf("Call-weighted average travel time: %.2f min\n", sol.objective)

Set parameter Username
Set parameter LicenseID to value 2857222
Academic license - for non-commercial use only - expires 2027-08-25
Status: OPTIMAL | solve time: 0.1 s
Call-weighted average travel time: 6.80 min


#### 6. Results

In [8]:
selected_sites = candidates[sol.selected, :]
selected_sites.n_areas_served = [count(==(j), sol.assigned) for j in sol.selected]
selected_sites.calls_served   = [sum(w[sol.assigned .== j]) for j in sol.selected]
selected_sites

Row,site_id,name,address,lat,lon,snap_miles,n_areas_served,calls_served
,String7,String,String31,Float64,Float64,Float64,Int64,Float64
1,FH_002,Engine 226,409 State Street,40.6871,-73.9828,0.0370268,4,24204.0
2,FH_019,Engine 230,701 Park Avenue,40.6981,-73.9475,0.0388862,7,35850.0
3,FH_034,Battalion 44/Engine 231/Ladder 120,107 Watkins Street,40.6706,-73.9079,0.0328743,9,69121.0
4,FH_046,Engine 282/Ladder 148,4210 12th Ave.,40.6402,-73.9908,0.0273946,9,35254.0
5,FH_057,Engine 254/Ladder 153,901 Avenue U,40.5985,-73.962,0.0206469,8,38062.0


In [9]:
assignment = DataFrame(
    area_id   = I_areas,
    weight    = w,
    site_id   = J_sites[sol.assigned],
    site_name = candidates.name[sol.assigned],
    minutes   = [t[sol.assigned[i], i] for i in eachindex(I_areas)],
)
sort(assignment, :minutes, rev=true)

Row,area_id,weight,site_id,site_name,minutes
,String,Float64,String7,String,Float64
1,11234,4959.0,FH_057,Engine 254/Ladder 153,19.3097
2,11209,4003.0,FH_046,Engine 282/Ladder 148,11.642
3,11224,6650.0,FH_057,Engine 254/Ladder 153,11.3214
4,11210,4008.0,FH_057,Engine 254/Ladder 153,10.8094
5,11236,6522.0,FH_034,Battalion 44/Engine 231/Ladder 120,10.5857
6,11203,7205.0,FH_034,Battalion 44/Engine 231/Ladder 120,10.2775
7,11239,2037.0,FH_034,Battalion 44/Engine 231/Ladder 120,10.2775
8,11222,2887.0,FH_019,Engine 230,10.0115
9,11208,9873.0,FH_034,Battalion 44/Engine 231/Ladder 120,9.92454


In [10]:
# Score any set of sites using the same objective as the optimization model.
function score_layout(t, weights, selected)
    nearest = vec(minimum(t[selected, :]; dims=1))
    average = sum(weights .* nearest) / sum(weights)
    return (average=average, worst_zip=maximum(nearest))
end

# Baseline 1: Start with the ZIPs with the most calls.
# For each of the top P ZIPs, pick its nearest unused candidate.
function busiest_first(t, weights, P)
    remaining = collect(axes(t, 1))
    selected = Int[]
    busiest_zips = sortperm(weights; rev=true)[1:P]

    for i in busiest_zips
        position = argmin(t[remaining, i])
        push!(selected, remaining[position])
        deleteat!(remaining, position)
    end

    return selected
end

# Baseline 2: Add one site at a time.
# At each step, pick the addition that gives the best borough-wide average.
function greedy_add(t, weights, P)
    remaining = collect(axes(t, 1))
    selected = Int[]
    nearest = fill(Inf, size(t, 2))

    for _ in 1:P
        scores = [sum(weights .* min.(nearest, t[j, :])) for j in remaining]
        position = argmin(scores)
        site = remaining[position]

        push!(selected, site)
        nearest = min.(nearest, t[site, :])
        deleteat!(remaining, position)
    end

    return selected
end

layouts = [busiest_first(t, w, P), greedy_add(t, w, P), sol.selected]
scores = [score_layout(t, w, selected) for selected in layouts]

# Confirm that our evaluation reproduces the solver's objective.
@assert isapprox(scores[3].average, sol.objective; atol=1e-6)

comparison = DataFrame(
    method = ["Busiest-first", "Greedy", "Optimization"],
    avg_minutes = [round(s.average; digits=2) for s in scores],
    worst_zip_minutes = [round(s.worst_zip; digits=2) for s in scores],
    selected_sites = [join(J_sites[selected], ", ") for selected in layouts],
)

display(comparison)

saved = scores[2].average - scores[3].average
@printf("Optimization saves %.1f seconds per call (%.1f%%) vs. greedy.\n",
        60 * saved, 100 * saved / scores[2].average)

Row,method,avg_minutes,worst_zip_minutes,selected_sites
,String,Float64,Float64,String
1,Busiest-first,8.83,24.3,"FH_036, FH_035, FH_004, FH_039, FH_043"
2,Greedy,7.13,25.25,"FH_041, FH_055, FH_034, FH_019, FH_005"
3,Optimization,6.8,19.31,"FH_002, FH_019, FH_034, FH_046, FH_057"


Optimization saves 19.8 seconds per call (4.6%) vs. greedy.


#### 7. Save outputs

In [ ]:
mkpath(OUTPUT_DIR)
CSV.write(joinpath(OUTPUT_DIR, "selected_sites_P$(P).csv"), selected_sites)
CSV.write(joinpath(OUTPUT_DIR, "area_assignment_P$(P).csv"), assignment)
println("Wrote results to $(OUTPUT_DIR)/")